In [2]:
import os
import glob
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.spatial.distance import pdist
from tqdm.auto import tqdm  # 自动适配 Jupyter 的进度条
from rdkit import Chem
from rdkit import RDLogger

# 抑制第三方库的底层警告
RDLogger.DisableLog('rdApp.*')  

# =============================================================================
# 配置路径与超参数 (仅扫描，不输出文件)
# =============================================================================
INPUT_DATA_ROOT = '/home/c319/miniconda3/envs/PhiStone/project/GNN/CHEMCUT/datas/train_datas/PDBbind' 
MAX_LIGAND_SPAN = 20.0        
NUM_WORKERS = 28              

# =============================================================================
# 核心辅助函数
# =============================================================================
def analyze_ligand_status(sdf_path):
    """分析配体状态，返回状态码"""
    suppl = Chem.SDMolSupplier(sdf_path, removeHs=True, sanitize=False)
    try:
        mol = next(suppl)
    except (StopIteration, Exception):
        return "rdkit_parse_error", None

    if mol is None or mol.GetNumHeavyAtoms() == 0:
        return "empty_or_no_heavy_atoms", None

    # 1. 连通性检查 (单一片段)
    if len(Chem.GetMolFrags(mol)) > 1:
        return "multiple_fragments", None

    # 2. 有机物检查 (至少含有一个碳原子)
    has_carbon = any(atom.GetAtomicNum() == 6 for atom in mol.GetAtoms())
    if not has_carbon:
        return "no_carbon_inorganic", None

    # 3. 几何跨度检查
    conf = mol.GetConformer(0)
    pos = conf.GetPositions()
    if len(pos) > 1:
        max_dist = np.max(pdist(pos))
        if max_dist > MAX_LIGAND_SPAN:
            return "span_too_large", None

    return "success", mol

def calculate_minimum_enclosing_ball(mol_with_conf):
    """计算包含所有重原子的最小包围球球心，仅为了测试是否会报错跳过"""
    heavy_atom_indices = [a.GetIdx() for a in mol_with_conf.GetAtoms() if a.GetAtomicNum() > 1]
    if not heavy_atom_indices:
        return None
        
    conf = mol_with_conf.GetConformer(0)
    points = np.array([list(conf.GetAtomPosition(i)) for i in heavy_atom_indices])
    
    center = np.mean(points, axis=0)
    for i in range(1, 151):
        dists_sq = np.sum((points - center)**2, axis=1)
        furthest_idx = np.argmax(dists_sq)
        p_i = points[furthest_idx]
        step = 1.0 / (i + 1)
        center = (1.0 - step) * center + step * p_i
        
    return center

# =============================================================================
# 多进程 Worker 函数 (纯扫描模式)
# =============================================================================
def scan_single_complex(complex_dir):
    stats = {
        "total_scanned": 0, "success": 0, "rdkit_parse_error": 0,
        "empty_or_no_heavy_atoms": 0, "multiple_fragments": 0,
        "no_carbon_inorganic": 0, "span_too_large": 0,
        "meb_calc_fail": 0, "missing_pocket_pdb": 0
    }
    
    sdf_files = glob.glob(os.path.join(complex_dir, "*_ligand.sdf"))
    if not sdf_files:
        return stats
        
    for sdf_path in sdf_files:
        stats["total_scanned"] += 1
        
        # 验证对应的 pocket.pdb 是否存在
        pocket_path = sdf_path.replace('_ligand.sdf', '_pocket.pdb')
        if not os.path.exists(pocket_path):
            stats["missing_pocket_pdb"] += 1
            continue
            
        # 验证 RDKit 质检规则
        status_code, lig_mol = analyze_ligand_status(sdf_path)
        if status_code != "success":
            stats[status_code] += 1
            continue
            
        # 验证中心计算是否正常
        center_coord = calculate_minimum_enclosing_ball(lig_mol)
        if center_coord is None:
            stats["meb_calc_fail"] += 1
            continue
            
        stats["success"] += 1
        
    return stats

# =============================================================================
# 主执行单元
# =============================================================================
print(f"正在递归检索 PDBbind 数据集 (路径: {INPUT_DATA_ROOT}) ...")
all_sdf_files = glob.glob(os.path.join(INPUT_DATA_ROOT, '**', '*_ligand.sdf'), recursive=True)
all_complex_dirs = list(set([os.path.dirname(f) for f in all_sdf_files]))

print(f"共发现 {len(all_complex_dirs)} 个包含配体的文件夹。开始进行安全空跑 (Dry Run)...\n")

global_stats = {
    "total_scanned": 0, "success": 0, "rdkit_parse_error": 0,
    "empty_or_no_heavy_atoms": 0, "multiple_fragments": 0,
    "no_carbon_inorganic": 0, "span_too_large": 0,
    "meb_calc_fail": 0, "missing_pocket_pdb": 0
}

# 开启多进程扫描
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = [executor.submit(scan_single_complex, d) for d in all_complex_dirs]
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="扫描进度"):
        try:
            local_stats = future.result()
            for key in global_stats:
                global_stats[key] += local_stats.get(key, 0)
        except Exception as e:
            pass
            
print("\n" + "="*70)
print("�� PDBbind 过滤策略摸底报告 (纯检测，无文件被修改)")
print("="*70)
print(f"�� 总计扫描的配体数目      : {global_stats['total_scanned']}")
print(f"✅ 预期保留的高质量数目    : {global_stats['success']}")
print(f"❌ 预期将被拦截的不合格数  : {global_stats['total_scanned'] - global_stats['success']}")
print("-" * 70)
print("⚠️ 预期不合格的具体原因分布:")
print(f"1. 几何跨度大于 {MAX_LIGAND_SPAN}Å  : {global_stats['span_too_large']} (超大分子、多肽等)")
print(f"2. 断裂/多片段组成的配体: {global_stats['multiple_fragments']} (网络结构、分散溶剂)")
print(f"3. 无机物/不含碳原子    : {global_stats['no_carbon_inorganic']} (游离金属簇、无机盐)")
print(f"4. 缺失对应 pocket 文件 : {global_stats['missing_pocket_pdb']} (数据不匹配)")
print(f"5. RDKit 加载解析失败   : {global_stats['rdkit_parse_error']} (化学价态不合法/SDF损坏)")
print(f"6. 空白或无重原子分子   : {global_stats['empty_or_no_heavy_atoms']} (如仅含有H或解析为空)")
print(f"7. 中心计算失败(MEB)    : {global_stats['meb_calc_fail']} (极少见)")
print("="*70)
print("�� 提示: 若某些维度的拦截数量远超你的预期(例如 span_too_large 拦截过多)，你可以考虑调宽对应的阈值参数再进行正式清洗。")

正在递归检索 PDBbind 数据集 (路径: /home/c319/miniconda3/envs/PhiStone/project/GNN/CHEMCUT/datas/train_datas/PDBbind) ...
共发现 29263 个包含配体的文件夹。开始进行安全空跑 (Dry Run)...



扫描进度:   0%|          | 0/29263 [00:00<?, ?it/s]


�� PDBbind 过滤策略摸底报告 (纯检测，无文件被修改)
�� 总计扫描的配体数目      : 29263
✅ 预期保留的高质量数目    : 25288
❌ 预期将被拦截的不合格数  : 3975
----------------------------------------------------------------------
⚠️ 预期不合格的具体原因分布:
1. 几何跨度大于 20.0Å  : 3884 (超大分子、多肽等)
2. 断裂/多片段组成的配体: 81 (网络结构、分散溶剂)
3. 无机物/不含碳原子    : 10 (游离金属簇、无机盐)
4. 缺失对应 pocket 文件 : 0 (数据不匹配)
5. RDKit 加载解析失败   : 0 (化学价态不合法/SDF损坏)
6. 空白或无重原子分子   : 0 (如仅含有H或解析为空)
7. 中心计算失败(MEB)    : 0 (极少见)
�� 提示: 若某些维度的拦截数量远超你的预期(例如 span_too_large 拦截过多)，你可以考虑调宽对应的阈值参数再进行正式清洗。
